<a href="https://colab.research.google.com/github/guillaumevalette2-hash/mse_gh/blob/main/QP_glouton_358.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import numpy as np
from itertools import product, combinations_with_replacement
from math import factorial
from collections import Counter
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import roc_auc_score
from sklearn.datasets import load_digits
from sklearn.svm import SVC
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import GridSearchCV
from scipy.optimize import minimize, LinearConstraint

# ══════════════════════════════════════════════════════════════════════════════
# PARAMÈTRES
# ══════════════════════════════════════════════════════════════════════════════
params = {
    "neg_digits":  [3, 5],
    "pos_digits":  [8],
    "img_size":    4,          # 8×8 -> 4×4  (dim ambiante 16)
    "n_train":     40,
    "n_test":      200,
    "n_unlabeled": 800,
    "seed":        123645,
    "seeds":{42: 8617,43: 9187 },
    "weights":        {0: 1e-4, 1: 1.0, 2: 0.0, 3: 0.0},   # poids QP complet
    "weights_screen": {0: 1e-4, 1: 1.0},                   # poids screening (moins cher)
    "lambda_G":       1e-9,
    "qp_margin":      1.0,
    "thres1":         1e-12,
    "thres2":         1e6,
    "const_pen":      0.0,
    "n_G":            600,      # taille de l'échantillon cloud pour les moments

    "deg_start":   3,           # degré ambiant de départ
    "new_deg":     4,           # degré à ajouter par étape gloutonne
    "dict_size":   400,         # taille du dictionnaire de candidats degré new_deg
    "n_select":    150,         # nombre retenu après screening H
}

# ══════════════════════════════════════════════════════════════════════════════
# DONNÉES : sklearn digits, {3,5} vs {8}, réduits 8×8 -> 4×4
# ══════════════════════════════════════════════════════════════════════════════
def load_digits_binary(neg_list, pos_list, img_size, n_train, n_test, n_unlabeled, seed=params["seeds"][42]):
    data = load_digits()
    X64, y_raw = data.images.astype(float), data.target.astype(int)
    keep = set(neg_list) | set(pos_list)
    mask = np.isin(y_raw, list(keep))
    X64 = X64[mask]; y_raw = y_raw[mask]
    y = np.where(np.isin(y_raw, list(pos_list)), 1.0, -1.0)
    if 8 % img_size != 0:
        raise ValueError(f"img_size={img_size} ne divise pas 8")
    b = 8 // img_size
    Xs = X64.reshape(-1, img_size, b, img_size, b).mean(axis=(2, 4))
    Xs = Xs.reshape(-1, img_size * img_size)
    Xs = (Xs - Xs.mean(0)) / (Xs.std(0) + 1e-8)
    rng = np.random.default_rng(seed)
    idx_neg = np.where(y < 0)[0]; idx_pos = np.where(y > 0)[0]
    rng.shuffle(idx_neg); rng.shuffle(idx_pos)
    n_tr_c = n_train // 2; n_te_c = n_test // 2
    tr = np.concatenate([idx_neg[:n_tr_c], idx_pos[:n_tr_c]])
    te = np.concatenate([idx_neg[n_tr_c:n_tr_c + n_te_c], idx_pos[n_tr_c:n_tr_c + n_te_c]])
    used = set(tr) | set(te)
    unlab_pool = np.array([i for i in range(len(Xs)) if i not in used])
    if len(unlab_pool) > n_unlabeled:
        unlab = rng.choice(unlab_pool, n_unlabeled, replace=False)
    else:
        unlab = unlab_pool
    rng.shuffle(tr); rng.shuffle(te); rng.shuffle(unlab)
    return Xs[tr], y[tr], Xs[te], y[te], Xs[unlab], y[unlab]


# ══════════════════════════════════════════════════════════════════════════════
# QP SOBOLEV — solveur générique
# ══════════════════════════════════════════════════════════════════════════════
def solve_qp(A, y, G, margin):
    n, k = A.shape
    Gr = G + 1e-12 * np.eye(k)
    con = LinearConstraint(np.diag(y) @ A, lb=margin, ub=np.inf)
    try:    c0 = np.linalg.lstsq(A, 1.5 * margin * y, rcond=None)[0]
    except Exception: c0 = np.zeros(k)
    res = minimize(lambda c: c @ Gr @ c, c0, jac=lambda c: 2 * Gr @ c,
                   constraints=[con], method='SLSQP',
                   options={'maxiter': 500, 'ftol': 1e-11})
    c = res.x
    marge_eff = float(np.min(y * (A @ c)))
    return c, marge_eff >= margin - 1e-4, marge_eff


# ══════════════════════════════════════════════════════════════════════════════
# BASELINE AMBIANTE SIMPLE (référence historique, PolynomialFeatures standard)
# ══════════════════════════════════════════════════════════════════════════════
def classify_poly_qp(X_tr, y_tr, X_te, y_te, X_cloud, deg, weights,
                     thres1, thres2, const_pen, qp_margin, n_G, titre=""):
    n, dd = X_tr.shape
    lo = X_cloud.min(axis=0); hi = X_cloud.max(axis=0)
    span = np.where(hi - lo > 1e-12, hi - lo, 1.0)
    def to_u(X): return 2 * (X - lo) / span - 1.0
    SC = 2.0 / span

    poly = PolynomialFeatures(degree=deg, include_bias=False)
    Phi_tr = poly.fit_transform(to_u(X_tr))
    powers = poly.powers_; n_feat = Phi_tr.shape[1]

    def pderiv(U, coefs, dims=()):
        P = powers.astype(float).copy(); m = coefs.astype(float).copy()
        for dm in dims: m = m * P[:, dm]; P[:, dm] -= 1
        v = m != 0
        return np.zeros(len(U)) if not v.any() else (U[:, None, :] ** P[None, v, :]).prod(2) @ m[v]

    rng = np.random.default_rng(params["seeds"][43])
    idx = rng.choice(len(X_cloud), min(n_G, len(X_cloud)), replace=False)
    U_G = to_u(X_cloud[idx]); nG = len(idx)
    PHI = poly.transform(U_G)
    E = np.eye(n_feat)
    w0 = weights.get(0, 0.); w1 = weights.get(1, 0.)
    G = np.zeros((n_feat, n_feat))
    if w0: G += w0 * (PHI.T @ PHI) / nG
    if w1:
        GG = np.zeros((nG, n_feat, dd))
        for j in range(n_feat):
            for a in range(dd):
                if powers[j, a] > 0:
                    GG[:, j, a] = SC[a] * pderiv(U_G, E[j], (a,))
        G += w1 * np.einsum('xik,xjk->ij', GG, GG) / nG
    G += 1e-9 * np.eye(n_feat)

    mean_phi = PHI.mean(axis=0)
    s_g, V_g = np.linalg.eigh(G)
    keep = (s_g > thres1) & (s_g < thres2)
    if keep.sum() == 0:
        raise ValueError("bande spectrale vide")
    T = V_g[:, keep] / np.sqrt(s_g[keep]); r = int(keep.sum())
    A_qp = np.hstack([(Phi_tr - mean_phi) @ T, np.ones((n, 1))])
    G_qp = np.eye(r + 1); G_qp[-1, -1] = const_pen
    sol, feas, marge = solve_qp(A_qp, y_tr, G_qp, qp_margin)
    coef = T @ sol[:r]; off = sol[-1] - mean_phi @ coef

    f_te = poly.transform(to_u(X_te)) @ coef + off
    f_tr = Phi_tr @ coef + off
    mse_te = float(np.mean((f_te - y_te) ** 2))
    normH = float(np.sqrt(max(sol[:r] @ sol[:r], 0)))
    def acc(f, y):
        sg = np.sign(f)
        return float(np.mean(np.where(sg == 0, 0.5, (sg == np.sign(y)).astype(float))))
    try:    auc = roc_auc_score((y_te > 0).astype(int), f_te)
    except Exception: auc = float('nan')
    print(f"  [{titre}] deg={deg} feat={n_feat} rang={r} | faisable={feas} marge={marge:.3f} "
          f"‖u‖_H={normH:.4f} MSE_te={mse_te:.4f} | "
          f"acc_tr={acc(f_tr,y_tr):.3f} acc_te={acc(f_te,y_te):.4f} AUC={auc:.4f}")
    return {'acc_te': acc(f_te, y_te), 'auc': auc, 'mse_te': mse_te, 'normH': normH}

def classify_ridge(X_tr, y_tr, X_te, y_te, deg, titre=""):
    poly = PolynomialFeatures(degree=deg)
    r = Ridge(alpha=1e-8).fit(poly.fit_transform(X_tr), y_tr)
    f = r.predict(poly.transform(X_te))
    a = np.mean(np.sign(f) == np.sign(y_te))
    mse = float(np.mean((f - y_te) ** 2))
    try:    auc = roc_auc_score((y_te > 0).astype(int), f)
    except Exception: auc = float('nan')
    print(f"  [{titre}] Ridge deg={deg} : acc_te={a:.4f} MSE_te={mse:.4f} AUC={auc:.4f}")
    return {'acc_te': a, 'auc': auc, 'mse_te': mse}


# ══════════════════════════════════════════════════════════════════════════════
# ARCHITECTURE "POOL DE MOMENTS" — TOUT (Gram QP, corrélation H de sélection,
# NORMES des candidats du dictionnaire) est dérivé du MÊME pool, calculé une
# seule fois par appel. Aucune norme n'est recalculée "à part".
#
#   - une dérivée de monôme EST un monôme (à un coefficient scalaire près) :
#     réduction algébrique de l'exposant + cache d'évaluation par exposant
#     DISTINCT (MonomialCache) — pas de recalcul redondant ;
#   - pas de tenseur dense (dd,dd,dd) : boucle sur les combos DISTINCTS de dims
#     à multiplicité près (C(dd+o-1,o), petit même en dimension ambiante pleine) ;
#   - validé à 1e-15/1e-16 près contre les calculs directs (voir historique).
# ══════════════════════════════════════════════════════════════════════════════
def _multiplicity(combo):
    c = Counter(combo); m = factorial(len(combo))
    for v in c.values(): m //= factorial(v)
    return m

def poly_eval_from_powers(U, powers):
    n = U.shape[0]; nf = powers.shape[0]
    Phi = np.ones((n, nf))
    for a in range(U.shape[1]):
        e = powers[:, a]
        if not np.any(e): continue
        maxe = int(e.max())
        pw = np.empty((maxe + 1, n)); pw[0] = 1.0
        if maxe >= 1: pw[1] = U[:, a]
        for p in range(2, maxe + 1): pw[p] = pw[p - 1] * U[:, a]
        Phi *= pw[e].T
    return Phi

class MonomialCache:
    """Cache des évaluations U^alpha (vecteur (nG,)) par tuple d'exposants."""
    def __init__(self, U_G):
        self.U_G = U_G
        self.cache = {}
    def eval(self, alpha):
        key = tuple(int(x) for x in alpha)
        if key not in self.cache:
            v = np.ones(self.U_G.shape[0])
            for d, e in enumerate(key):
                if e: v = v * self.U_G[:, d] ** e
            self.cache[key] = v
        return self.cache[key]

def _reduce_monomial(p, combo_count):
    coeff = 1.0
    r = p.copy()
    for a, k in combo_count.items():
        if r[a] < k:
            return 0.0, None
        for i in range(k):
            coeff *= (r[a] - i)
        r[a] -= k
    return coeff, r

def build_deriv_pool(U_G, powers_pool, weights, SC, max_order=3):
    """pool[0] = PHI_pool (nG, n_pool) si w0 utilisé.
    pool[o] = liste de (mult, GG_combo) pour o=1..max_order.
    `weights` = union des ordres nécessaires (screening + refit)."""
    nG, dd = U_G.shape
    n_pool = powers_pool.shape[0]
    mc = MonomialCache(U_G)

    pool = {}
    if weights.get(0, 0.):
        PHI = np.zeros((nG, n_pool))
        for j in range(n_pool):
            PHI[:, j] = mc.eval(powers_pool[j])
        pool[0] = PHI

    for o in range(1, max_order + 1):
        if not weights.get(o, 0.):
            continue
        entries = []
        for combo in combinations_with_replacement(range(dd), o):
            mult = _multiplicity(combo)
            combo_count = Counter(combo)
            sc_factor = np.prod([SC[a] for a in combo])
            GG = np.zeros((nG, n_pool))
            for j in range(n_pool):
                coeff, alpha = _reduce_monomial(powers_pool[j], combo_count)
                if coeff:
                    GG[:, j] = sc_factor * coeff * mc.eval(alpha)
            entries.append((mult, GG))
        pool[o] = entries
    return pool

def gram_from_pool(pool, weights, idx_rows, idx_cols, nG, lambda_G=0.0, add_reg=False):
    """Bloc de Gram H(cloud) restreint à idx_rows × idx_cols — pur produit
    matriciel sur le pool déjà calculé."""
    G = np.zeros((len(idx_rows), len(idx_cols)))
    w0 = weights.get(0, 0.)
    if w0 and 0 in pool:
        PHI = pool[0]
        G += w0 * (PHI[:, idx_rows].T @ PHI[:, idx_cols]) / nG
    for o in range(1, 4):
        wo = weights.get(o, 0.)
        if not wo or o not in pool:
            continue
        for mult, GG in pool[o]:
            G += wo * mult * (GG[:, idx_rows].T @ GG[:, idx_cols]) / nG
    if add_reg:
        G += lambda_G * np.eye(len(idx_rows))
    return G

def diag_from_pool(pool, weights, idx, nG):
    """Normes ‖φ_i‖²_H pour i in idx — MÊME pool, mêmes formules que Gram/cross.
    C'est la fonction unique utilisée partout où une norme de monôme du
    dictionnaire est nécessaire (plus de calcul indépendant)."""
    d = np.zeros(len(idx))
    w0 = weights.get(0, 0.)
    if w0 and 0 in pool:
        d += w0 * np.mean(pool[0][:, idx] ** 2, axis=0)
    for o in range(1, 4):
        wo = weights.get(o, 0.)
        if not wo or o not in pool:
            continue
        for mult, GG in pool[o]:
            d += wo * mult * np.mean(GG[:, idx] ** 2, axis=0)
    return d


# ══════════════════════════════════════════════════════════════════════════════
# QP sur un ensemble de monômes arbitraire — Gram ET centrage viennent du pool
# ══════════════════════════════════════════════════════════════════════════════
def fit_qp_from_pool(X_tr, y_tr, X_te, y_te, powers, lo, span, pool, idx_final,
                     weights, lambda_G, thres1, thres2, const_pen, qp_margin, nG,
                     titre=""):
    n = X_tr.shape[0]
    def to_u(X): return 2 * (X - lo) / span - 1.0
    Phi_tr = poly_eval_from_powers(to_u(X_tr), powers)

    G = gram_from_pool(pool, weights, idx_final, idx_final, nG, lambda_G, add_reg=True)
    # centrage EXACT : moyenne de PHI sur le même échantillon cloud que le pool
    if 0 in pool:
        mean_phi = pool[0][:, idx_final].mean(axis=0)
    else:
        mean_phi = np.zeros(len(idx_final))

    s_g, V_g = np.linalg.eigh(G)
    keep = (s_g > thres1) & (s_g < thres2)
    if keep.sum() == 0:
        raise ValueError("bande spectrale vide")
    T = V_g[:, keep] / np.sqrt(s_g[keep]); r = int(keep.sum())
    A_qp = np.hstack([(Phi_tr - mean_phi) @ T, np.ones((n, 1))])
    G_qp = np.eye(r + 1); G_qp[-1, -1] = const_pen
    sol, feas, marge = solve_qp(A_qp, y_tr, G_qp, qp_margin)
    coef = T @ sol[:r]; off = sol[-1] - mean_phi @ coef

    f_te = poly_eval_from_powers(to_u(X_te), powers) @ coef + off
    f_tr = Phi_tr @ coef + off
    mse_te = float(np.mean((f_te - y_te) ** 2))
    normH = float(np.sqrt(max(sol[:r] @ sol[:r], 0)))
    def acc(f, y):
        sg = np.sign(f)
        return float(np.mean(np.where(sg == 0, 0.5, (sg == np.sign(y)).astype(float))))
    try:    auc = roc_auc_score((y_te > 0).astype(int), f_te)
    except Exception: auc = float('nan')
    print(f"  [{titre}] n_feat={len(idx_final)} rang={r} | faisable={feas} marge={marge:.3f} "
          f"‖u‖_H={normH:.4f} MSE_te={mse_te:.4f} | "
          f"acc_tr={acc(f_tr,y_tr):.3f} acc_te={acc(f_te,y_te):.4f} AUC={auc:.4f}")
    return {'coef': coef, 'off': off, 'mse_te': mse_te, 'normH': normH,
            'acc_te': acc(f_te, y_te), 'auc': auc}


# ══════════════════════════════════════════════════════════════════════════════
# ÉTAPE GLOUTONNE — un seul pool (actifs ∪ candidats), tout en découle :
# Gram QP, corrélation H de sélection, ET normes des candidats.
# ══════════════════════════════════════════════════════════════════════════════
def greedy_add_degree(X_train, y_train, X_all, X_test, y_test,
                      active_powers, prev_coef, prev_normH,
                      new_deg, dict_size, n_select,
                      weights, weights_screen, lambda_G,
                      thres1, thres2, const_pen, qp_margin, n_G_sample,
                      seed, max_order=3):
    dd = X_train.shape[1]
    rng = np.random.default_rng(seed)

    full_combos_all = list(combinations_with_replacement(range(dd), new_deg))
    full_combos = full_combos_all
    if len(full_combos) > dict_size:
        chosen = rng.choice(len(full_combos), dict_size, replace=False)
        full_combos = [full_combos[i] for i in chosen]
    cand_powers = np.zeros((len(full_combos), dd), dtype=int)
    for i, combo in enumerate(full_combos):
        for a in combo: cand_powers[i, a] += 1
    print(f"\n── Degré {new_deg} : dictionnaire {cand_powers.shape[0]} monômes "
          f"(sur {len(full_combos_all)} possibles)")

    lo = X_all.min(axis=0); hi = X_all.max(axis=0)
    span = np.where(hi - lo > 1e-12, hi - lo, 1.0)
    SC = 2.0 / span
    rng_g = np.random.default_rng(params["seeds"][43])
    idx_g = rng_g.choice(len(X_all), min(n_G_sample, len(X_all)), replace=False)
    U_G = 2 * (X_all[idx_g] - lo) / span - 1.0
    nG = len(idx_g)

    n_active = active_powers.shape[0]
    n_cand = cand_powers.shape[0]
    powers_pool = np.vstack([active_powers, cand_powers])
    idx_active = list(range(n_active))
    idx_cand = list(range(n_active, n_active + n_cand))

    needed_orders = set()
    for o in range(0, max_order + 1):
        if weights.get(o, 0.) or weights_screen.get(o, 0.):
            needed_orders.add(o)
    weights_union = {o: 1.0 for o in needed_orders}
    pool = build_deriv_pool(U_G, powers_pool, weights_union, SC, max_order=max_order)

    # ── screening : corrélation H (poids réduits), norme des candidats
    #    calculée avec la MÊME diag_from_pool (pas de calcul indépendant) ──
    Gc = gram_from_pool(pool, weights_screen, idx_cand, idx_active, nG)
    proj = Gc @ prev_coef
    diag_cand = diag_from_pool(pool, weights_screen, idx_cand, nG)
    cos_sim = proj / (np.sqrt(np.maximum(diag_cand, 1e-30)) * max(prev_normH, 1e-30))

    order_sel = np.argsort(-np.abs(cos_sim))[:n_select]
    print(f"   sélection : {len(order_sel)}/{n_cand} monômes "
          f"(|cos_H| min retenu = {np.abs(cos_sim[order_sel]).min():.4f}, "
          f"max = {np.abs(cos_sim[order_sel]).max():.4f})")
    print(f"   normes ‖φ‖_H des candidats sélectionnés : "
          f"min={np.sqrt(diag_cand[order_sel]).min():.4f} "
          f"max={np.sqrt(diag_cand[order_sel]).max():.4f}")

    idx_final = idx_active + [idx_cand[i] for i in order_sel]
    new_powers = powers_pool[idx_final]

    sol = fit_qp_from_pool(X_train, y_train, X_test, y_test, new_powers, lo, span,
                           pool, idx_final, weights, lambda_G, thres1, thres2,
                           const_pen, qp_margin, nG,
                           titre=f"QP actifs({n_active}) + {n_select} monômes deg={new_deg}")
    return new_powers, sol


# ══════════════════════════════════════════════════════════════════════════════
# EXPÉRIENCE
# ══════════════════════════════════════════════════════════════════════════════
X_train, y_train, X_test, y_test, X_unlab, y_unlab = load_digits_binary(
    params["neg_digits"], params["pos_digits"], params["img_size"],
    params["n_train"], params["n_test"], params["n_unlabeled"], seed=params["seed"])
X_all = np.vstack([X_train, X_unlab])
d = X_train.shape[1]
print(f"digits {params['neg_digits']} vs {params['pos_digits']}, "
      f"{params['img_size']}×{params['img_size']} -> dim {d}")
print(f"  train: {len(X_train)}  test: {len(X_test)}  unlabeled: {len(X_unlab)}")

print("\n" + "=" * 72)
print("BASELINE AMBIANTE (PolynomialFeatures, référence historique)")
print("=" * 72)
res_qp2 = classify_poly_qp(X_train, y_train, X_test, y_test, X_all, 2,
                           params["weights"], params["thres1"], params["thres2"],
                           params["const_pen"], params["qp_margin"], params["n_G"],
                           titre="QP ambiant deg=2")
res_ridge = classify_ridge(X_train, y_train, X_test, y_test, 3, titre="Ridge deg=3")

print("\n" + "=" * 72)
print(f"GLOUTON PAR MOMENTS : deg<={params['deg_start']} -> +{params['new_deg']}")
print("=" * 72)
poly0 = PolynomialFeatures(degree=params["deg_start"], include_bias=False)
poly0.fit(np.zeros((1, d)))
powers0 = poly0.powers_
res0 = classify_poly_qp(X_train, y_train, X_test, y_test, X_all, params["deg_start"],
                        params["weights"], params["thres1"], params["thres2"],
                        params["const_pen"], params["qp_margin"], params["n_G"],
                        titre=f"QP deg<={params['deg_start']} (point de départ glouton)")

# pour démarrer le glouton il faut coef+normH du départ -> on les recalcule
# via le pool directement (cohérence totale avec la suite)
lo0 = X_all.min(axis=0); hi0 = X_all.max(axis=0)
span0 = np.where(hi0 - lo0 > 1e-12, hi0 - lo0, 1.0)
SC0 = 2.0 / span0
rng_g0 = np.random.default_rng(params["seeds"][43])
idx_g0 = rng_g0.choice(len(X_all), min(params["n_G"], len(X_all)), replace=False)
U_G0 = 2 * (X_all[idx_g0] - lo0) / span0 - 1.0
needed0 = {o for o in range(4) if params["weights"].get(o, 0.)}
pool0 = build_deriv_pool(U_G0, powers0, {o: 1.0 for o in needed0}, SC0)
idx0 = list(range(powers0.shape[0]))
sol0 = fit_qp_from_pool(X_train, y_train, X_test, y_test, powers0, lo0, span0,
                        pool0, idx0, params["weights"], params["lambda_G"],
                        params["thres1"], params["thres2"], params["const_pen"],
                        params["qp_margin"], len(idx_g0),
                        titre=f"QP deg<={params['deg_start']} (via pool, cohérence)")

new_powers, sol1 = greedy_add_degree(
    X_train, y_train, X_all, X_test, y_test,
    active_powers=powers0, prev_coef=sol0['coef'], prev_normH=sol0['normH'],
    new_deg=params["new_deg"], dict_size=params["dict_size"], n_select=params["n_select"],
    weights=params["weights"], weights_screen=params["weights_screen"],
    lambda_G=params["lambda_G"], thres1=params["thres1"], thres2=params["thres2"],
    const_pen=params["const_pen"], qp_margin=params["qp_margin"], n_G_sample=params["n_G"],
    seed=params["seed"])

print("\n" + "=" * 72)
print("BASELINES RBF")
print("=" * 72)
param_grid_svm = {"C": [0.1, 1, 10, 100], "gamma": ["scale", 0.01, 0.1, 1]}
svm = GridSearchCV(SVC(kernel="rbf"), param_grid_svm, cv=5, n_jobs=-1)
svm.fit(X_train, y_train)
f_svm = svm.decision_function(X_test)
acc_svm = float(np.mean(np.sign(f_svm) == np.sign(y_test)))
mse_svm = float(np.mean((f_svm - y_test) ** 2))
try:    auc_svm = roc_auc_score((y_test > 0).astype(int), f_svm)
except Exception: auc_svm = float('nan')
print(f"  [SVM RBF]   best={svm.best_params_} | acc_te={acc_svm:.4f} MSE_te={mse_svm:.4f} AUC={auc_svm:.4f}")

param_grid_kr = {"alpha": [1e-3, 1e-2, 1e-1, 1.0], "gamma": [0.001, 0.01, 0.1, 1]}
kr = GridSearchCV(KernelRidge(kernel="rbf"), param_grid_kr, cv=5, n_jobs=-1)
kr.fit(X_train, y_train)
f_kr = kr.predict(X_test)
acc_kr = float(np.mean(np.sign(f_kr) == np.sign(y_test)))
mse_kr = float(np.mean((f_kr - y_test) ** 2))
try:    auc_kr = roc_auc_score((y_test > 0).astype(int), f_kr)
except Exception: auc_kr = float('nan')
print(f"  [Ridge RBF] best={kr.best_params_} | acc_te={acc_kr:.4f} MSE_te={mse_kr:.4f} AUC={auc_kr:.4f}")

print("\n" + "=" * 72)
print("BILAN")
print("=" * 72)
print(f"  QP ambiant deg=2                 : acc={res_qp2['acc_te']:.4f} MSE={res_qp2['mse_te']:.4f} "
      f"AUC={res_qp2['auc']:.4f} ‖u‖_H={res_qp2['normH']:.4f}")
print(f"  Ridge deg=3                       : acc={res_ridge['acc_te']:.4f} MSE={res_ridge['mse_te']:.4f} "
      f"AUC={res_ridge['auc']:.4f}")
print(f"  QP deg<={params['deg_start']} (via pool)         : acc={sol0['acc_te']:.4f} MSE={sol0['mse_te']:.4f} "
      f"AUC={sol0['auc']:.4f} ‖u‖_H={sol0['normH']:.4f}")
print(f"  QP deg<={params['deg_start']}+glouton deg={params['new_deg']} : acc={sol1['acc_te']:.4f} "
      f"MSE={sol1['mse_te']:.4f} AUC={sol1['auc']:.4f} ‖u‖_H={sol1['normH']:.4f}")
print(f"  SVM RBF                           : acc={acc_svm:.4f} MSE={mse_svm:.4f} AUC={auc_svm:.4f}")
print(f"  Ridge RBF                         : acc={acc_kr:.4f} MSE={mse_kr:.4f} AUC={auc_kr:.4f}")

digits [3, 5] vs [8], 4×4 -> dim 16
  train: 40  test: 200  unlabeled: 299

BASELINE AMBIANTE (PolynomialFeatures, référence historique)
  [QP ambiant deg=2] deg=2 feat=152 rang=152 | faisable=True marge=1.000 ‖u‖_H=0.8149 MSE_te=0.3735 | acc_tr=1.000 acc_te=0.9600 AUC=0.9949
  [Ridge deg=3] Ridge deg=3 : acc_te=0.8850 MSE_te=0.6890 AUC=0.9807

GLOUTON PAR MOMENTS : deg<=3 -> +4
  [QP deg<=3 (point de départ glouton)] deg=3 feat=968 rang=968 | faisable=True marge=1.000 ‖u‖_H=0.4567 MSE_te=0.2107 | acc_tr=1.000 acc_te=0.9600 AUC=0.9945
  [QP deg<=3 (via pool, cohérence)] n_feat=968 rang=968 | faisable=True marge=1.000 ‖u‖_H=0.4567 MSE_te=0.2107 | acc_tr=1.000 acc_te=0.9600 AUC=0.9945

── Degré 4 : dictionnaire 400 monômes (sur 3876 possibles)
   sélection : 150/400 monômes (|cos_H| min retenu = 0.0575, max = 0.1999)
   normes ‖φ‖_H des candidats sélectionnés : min=0.1101 max=0.7726
  [QP actifs(968) + 150 monômes deg=4] n_feat=1118 rang=1118 | faisable=True marge=1.000 ‖u‖_H=0.4367 MSE_